In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install cuml-cu12 --extra-index-url=https://pypi.nvidia.com

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    matthews_corrcoef,
)
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedGroupKFold,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib

In [ ]:
DIR_PATH = '/content/drive/MyDrive/1. Academics/ENEE408N/ENEE408N Project/sample of DREAMT dataset'
# DIR_PATH = '/data'
TRAIN_PATH = os.path.join(DIR_PATH, 'unbal_all_ppg_feat_extr_train.csv')
TEST_PATH = os.path.join(DIR_PATH, 'unbal_all_ppg_feat_extr_test.csv')

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

X_train = train_df.drop(['Label'], axis=1)
y_train = train_df['Label']
X_test = test_df.drop(['Label'], axis=1)
y_test = test_df['Label']

A = sum(y_train)
X = len(y_train)
print(f'Apnea count: {A}/{X} ({100*A/X:.2f}%)')
print(f'Nonapnea count: {X-A}/{X} ({100*(X-A)/X:.2f}%)')

# ML Models

In [ ]:
def select_scaler(model, dim_reduction):
    """Return appropriate scaler based on model and dim_reduction types."""
    if isinstance(dim_reduction, SelectKBest) and dim_reduction.score_func is chi2:
        return MinMaxScaler()
    if isinstance(model, (KNeighborsClassifier, SVC)):
        return StandardScaler()
    return None  # RandomForest and others need no scaling


def build_pipeline(model, scaler, dim_reduction, oversampler):
    """Build an ImbPipeline dynamically without if/else branching per combination.

    Steps order: oversampler -> (scaler?) -> (dim_reduction?) -> model
    """
    steps = [("oversampler", oversampler)]
    if scaler is not None:
        steps.append(("scaler", scaler))
    if dim_reduction is not None:
        steps.append(("dim_reduction", dim_reduction))
    steps.append(("model", model))
    return ImbPipeline(steps)


def param_grid(model_name, dim_reduction_name):
    """Return hyperparameter search space keyed by pipeline step name."""
    grid = {}

    model_grids = {
        "knn": {"model__n_neighbors": [3, 5, 7, 11]},
        "rf": {
            "model__n_estimators": [50, 100, 200],
            "model__max_depth": [None, 5, 10, 20],
        },
        "svm": {
            "model__C": [0.1, 1, 10, 100],
            "model__gamma": ["scale", "auto", 0.01, 0.001],
        },
    }
    grid.update(model_grids.get(model_name, {}))

    dim_reduction_grids = {
        "selectkbest_chi2": {"dim_reduction__k": [5, 10, 15, 20]},
        "selectkbest_f_classif": {"dim_reduction__k": [5, 10, 15, 20]},
        "pca": {"dim_reduction__n_components": [3, 5, 10, 15]},
    }
    grid.update(dim_reduction_grids.get(dim_reduction_name, {}))

    return grid


def model_report(pipeline, X_test, y_test, show=False):
    y_pred = pipeline.predict(X_test)

    metrics = {}
    metrics['accuracy'] = accuracy_score(y_test, y_pred)
    metrics['conf_mat'] = confusion_matrix(y_test, y_pred)
    metrics['classification_rep'] = classification_report(y_test, y_pred)

    if show:
        print("Accuracy:", metrics['accuracy'])
        print("Confusion matrix:\n", metrics['conf_mat'])
        print("Classification report:\n", metrics['classification_rep'])

    return metrics

## KNN

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=StandardScaler(),
    name='knn_3'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

KNN + Univariate

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=StandardScaler(),
    dim_reduction=SelectKBest(k=5),
    name='knn_3_univariate_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

KNN + Chi_square

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=MinMaxScaler(),
    dim_reduction=SelectKBest(score_func=chi2, k=5),
    name='knn_3_chi_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

KNN + PCA

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=StandardScaler(),
    dim_reduction=PCA(n_components=3),
    name='knn_3_pca_3'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

KNN + LDA

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=StandardScaler(),
    dim_reduction=LinearDiscriminantAnalysis(n_components=1),
    name='knn_3_lda_1'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

## RF

RF + Univariate

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_leaf=5
)
pipeline = model_pipeline(
    X_train,
    y_train,
    model=rf,
    scaler=StandardScaler(),
    dim_reduction=SelectKBest(k=5),
    name='rf_100w_univariate_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

RF + chi

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_leaf=5
)
pipeline = model_pipeline(
    X_train,
    y_train,
    model=rf,
    scaler=MinMaxScaler(),
    dim_reduction=SelectKBest(score_func=chi2, k=5),
    name='rf_100w_chi_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

RF + PCA

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_leaf=5
)
pipeline = model_pipeline(
    X_train,
    y_train,
    model=rf,
    scaler=StandardScaler(),
    dim_reduction=PCA(n_components=3),
    name='rf_100w_pca_3'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

RF + LDA

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_leaf=5
)
pipeline = model_pipeline(
    X_train,
    y_train,
    model=rf,
    scaler=StandardScaler(),
    dim_reduction=LinearDiscriminantAnalysis(n_components=1),
    name='rf_100w_lda_1'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

In [ ]:
import os
# Check it exists and see its size
path = os.path.join(DIR_PATH, 'rf_100w_lda_1.joblib')
print(os.path.exists(path))        # should be True
print(os.path.getsize(path), "bytes")

# Download
from google.colab import files
files.download(path)

## SVM

SVM + univariate

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=SVC(kernel='rbf'),
    scaler=StandardScaler(),
    dim_reduction=SelectKBest(k=5),
    name='svm_rbf_univariate_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

SVM + chi2

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=SVC(kernel='rbf'),
    scaler=MinMaxScaler(),
    dim_reduction=SelectKBest(score_func=chi2, k=5),
    name='svm_rbf_chi_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

SVM + PCA

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=SVC(kernel='rbf'),
    scaler=StandardScaler(),
    dim_reduction=PCA(n_components=3),
    name='svm_rbf_pca_3'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

SVM + LDA

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=SVC(kernel='rbf'),
    scaler=StandardScaler(),
    dim_reduction=LinearDiscriminantAnalysis(n_components=1),
    name='svm_rbf_lda_1'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

# Pipeline Helpers Demo & Tests

In [ ]:
# Demo: KNN + SelectKBest(chi2) end-to-end through build_pipeline
_demo_dim_red = SelectKBest(score_func=chi2, k=5)
_demo_model = KNeighborsClassifier(n_neighbors=3)
_demo_oversampler = RandomOverSampler(random_state=42)
_demo_scaler = select_scaler(_demo_model, _demo_dim_red)

_demo_pipeline = build_pipeline(
    model=_demo_model,
    scaler=_demo_scaler,
    dim_reduction=_demo_dim_red,
    oversampler=_demo_oversampler,
)
_demo_pipeline.fit(X_train, y_train)
print("Demo pipeline steps:", [name for name, _ in _demo_pipeline.steps])
_ = model_report(_demo_pipeline, X_test, y_test, show=True)

In [ ]:
# Tests: verify build_pipeline step names and order
def _test_build_pipeline():
    ros = RandomOverSampler(random_state=0)
    knn = KNeighborsClassifier()
    svm = SVC()
    rf = RandomForestClassifier()
    ss = StandardScaler()
    mm = MinMaxScaler()
    chi_sel = SelectKBest(score_func=chi2, k=5)
    f_sel = SelectKBest(k=5)
    pca = PCA(n_components=3)

    # Case 1: oversampler only + model -> 2 steps
    p = build_pipeline(rf, None, None, ros)
    assert [n for n, _ in p.steps] == ["oversampler", "model"], f"Case1 fail: {p.steps}"

    # Case 2: oversampler + scaler + model -> 3 steps
    p = build_pipeline(knn, ss, None, ros)
    assert [n for n, _ in p.steps] == ["oversampler", "scaler", "model"], f"Case2 fail: {p.steps}"

    # Case 3: oversampler + dim_reduction + model -> 3 steps
    p = build_pipeline(rf, None, pca, ros)
    assert [n for n, _ in p.steps] == ["oversampler", "dim_reduction", "model"], f"Case3 fail: {p.steps}"

    # Case 4: all present -> 4 steps
    p = build_pipeline(knn, ss, pca, ros)
    assert [n for n, _ in p.steps] == ["oversampler", "scaler", "dim_reduction", "model"], f"Case4 fail: {p.steps}"

    # select_scaler: chi2 -> MinMaxScaler
    assert isinstance(select_scaler(knn, chi_sel), MinMaxScaler), "chi2 scaler fail"

    # select_scaler: KNN + non-chi2 -> StandardScaler
    assert isinstance(select_scaler(knn, f_sel), StandardScaler), "knn scaler fail"

    # select_scaler: SVM -> StandardScaler
    assert isinstance(select_scaler(svm, f_sel), StandardScaler), "svm scaler fail"

    # select_scaler: RF -> None
    assert select_scaler(rf, f_sel) is None, "rf scaler fail"

    # param_grid contains expected keys
    g = param_grid("knn", "pca")
    assert "model__n_neighbors" in g, "knn param missing"
    assert "dim_reduction__n_components" in g, "pca param missing"

    g = param_grid("rf", "selectkbest_chi2")
    assert "model__n_estimators" in g
    assert "model__max_depth" in g
    assert "dim_reduction__k" in g

    g = param_grid("svm", "selectkbest_f_classif")
    assert "model__C" in g
    assert "model__gamma" in g
    assert "dim_reduction__k" in g

    print("All pipeline helper tests passed.")

_test_build_pipeline()

# Feedforward Neural Net